# 11-gauss-f — Lượt F: SVI trên nhãn cụm

**Câu hỏi**: lượt F học gì cho 5 khe lớp mới từ nhãn cụm, đi xa điểm khởi tạo đến đâu, có chạm vào khe lớp cũ không, và nếu dừng pipeline ngay sau lượt F thì tập test lớp mới được bao nhiêu (cột 11 của sổ mất điểm)? (D11, D12, D14, D23; QD24)

### Nguồn gốc

Mẫu cell 0 (`lab/README.vi.md` §11): in commit và trạng thái dirty, rồi đặt `XLA_FLAGS` ở chế độ GPU tất định trước mọi lần import jax.

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit ae132c1 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `ae132c1`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 11 ở r0 s4

Chạy lại bước 11 của `gc10_v2_ft_r0_s4` từ ảnh chụp 10 bằng `replay.rerun`, rồi so khoá `params_f` (μ và Σ của 10 khe) với ảnh chụp 11. Đây là số thực, nên cho phép lệch tới 1e-6 × max|giá trị|. Sau đó so 5 khe cũ của `params_f` với `params0` từng bit, và đếm dòng log có `early_stop_flag`.

Dự đoán [SUY LUẬN]:
- `rerun` OK 1/1 khoá, max|diff| = 0: SVI chạy ở chế độ GPU tất định, và ở `06` §1.1 bước 06 đã trùng từng bit dù driver đổi;
- 5 khe cũ trùng `params0` từng bit: không ảnh nào của pool mang nhãn 0–4, nên khe cũ không nhận gradient, và Adam với gradient bằng 0 cho bước cập nhật bằng 0 (D12);
- không có dòng `early_stop_flag`: lượt F không dừng ở cả 15 ảnh chụp (`07` §3.1), vì ngưỡng của nhánh F là det cũ / 2 = 0.185 ở r0 s4.

In [2]:
# 1.1 rerun step 11 (pass F, 1000 SVI steps on the GPU) and compare with the snapshot; old slots vs params0
import io, contextlib, time
import numpy as np

RUN = "gc10_v2_ft_r0_s4"
buf, t0 = io.StringIO(), time.time()
with contextlib.redirect_stdout(buf):
    out, ok = replay.rerun(RUN, "11")
log = buf.getvalue()
print("\n".join(l for l in log.splitlines() if l.startswith(("rerun", "  OK", "  DIFF"))))
print(f"time {time.time() - t0:.0f} s | log lines with early_stop_flag: {log.count('early_stop_flag')}")

p0 = replay.snapshots(RUN)["06"]["out"]["params0"]
pf = out["params_f"]
for k in ("class_means", "class_covs"):
    print(f"{k}: old slots 0-4 bit-identical to params0: {np.array_equal(pf[k][:5], p0[k][:5])} | "
          f"new slots 5-9 max|change from init| {np.abs(pf[k][5:] - p0[k][5:]).max():.4f}")

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1000/1000 [00:59<00:00, 16.86it/s]

rerun step 11 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
  OK   params_f     dict                   max|diff| 0
rerun step 11: OK (1/1 keys)
time 69 s | log lines with early_stop_flag: 0
class_means: old slots 0-4 bit-identical to params0: True | new slots 5-9 max|change from init| 0.0324
class_covs: old slots 0-4 bit-identical to params0: True | new slots 5-9 max|change from init| 0.0428


- **Kết quả**:
  - `rerun` bước 11 ở r0 s4: OK 1/1 khoá, max|diff| = 0, dù driver GPU đổi (617.42 → 621.01). 1000 bước SVI chạy 59 s; cả cell 70 s.
  - Không dòng log nào có `early_stop_flag`.
  - 5 khe cũ của `params_f` trùng `params0` từng bit, cả μ lẫn Σ.
  - 5 khe mới rời điểm khởi tạo (μ = 0, Σ = I) tối đa 0.0324 ở μ và 0.0428 ở Σ.
- **Ý nghĩa**:
  - Đúng như dự đoán ở cả ba ý. Lượt F tái lập từng bit; không dừng sớm; khe cũ không đổi.
  - μ của khe mới đi 0.0324, tức 97% ngân sách 0.0334, như khe cũ ở S0 (0.0329, `06` §2.2), nên lượt F nhận đủ ngân sách. Σ vượt ngân sách (0.0428 > 0.0334; ở S0 là 0.0357), vì Adam có thể bước xa hơn learning rate khi độ lớn gradient đổi nhanh (Kingma & Ba 2015, §2.1, cận trên khoảng 3.16 lần).

## 2. Code

### 2.1 Lượt F gọi gì, và nhánh F của luật dừng còn cách bao xa

In code của `s11_gauss_f`. Sau đó tính đúng như `calculate_metrics_on_covariances` khi `use_correct_scaling_factor=False` (nhánh của lượt F), bằng det float32 của JAX:
- dets[0] = det trung bình của 5 khe cũ (lấy từ `global_params` = `params0`);
- dets[1] = det trung bình của 5 khe mới, ở `params_f` (sau 1000 bước);
- sf = (offset + increment) / base = (5 + 5) / 5 = 2; dừng khi dets[0] < 1 và dets[1] < dets[0] / sf.
So thêm với ngưỡng của lượt S, dets[0] / √(5/10) (`07` §2.2), để biết nếu lượt F dùng luật của lượt S thì đã dừng chưa.

Dự đoán [SUY LUẬN]:
- dets[0] = 0.3707, ngưỡng F = 0.185, ngưỡng S = 0.524 (`07`);
- det trung bình của khe mới sau 1000 bước khoảng 0.3–0.4, vì khe mới nhận cùng ngân sách như khe cũ ở S0 (1.1). Giá trị đó cao hơn 0.185 (lượt F không dừng) nhưng thấp hơn 0.524: với luật của lượt S, lượt F đã dừng trước bước 1000.

In [3]:
# 2.1 the code of step 11; the early-stop branch of pass F evaluated on params0 and params_f
import inspect
import jax, jax.numpy as jnp
print(inspect.getsource(replay.s11_gauss_f))

det = lambda S: np.asarray(jax.vmap(jnp.linalg.det)(jnp.asarray(S)))      # float32, as mngmm does
d_old = det(p0["class_covs"][:5]); d_new = det(pf["class_covs"][5:])
print("det old slots (params0):", d_old.round(4), "| mean", round(float(d_old.mean()), 4))
print("det new slots (params_f, step 1000):", d_new.round(4), "| mean", round(float(d_new.mean()), 4))
thr_f, thr_s = d_old.mean() / 2.0, d_old.mean() / np.sqrt(5 / 10)
print(f"pass-F threshold {thr_f:.4f} -> stop: {d_new.mean() < thr_f} | pass-S threshold {thr_s:.4f} -> stop: {d_new.mean() < thr_s}")

def s11_gauss_f(st):
    """11 Pass F: SVI on the whole pool, cluster ids as labels, started from the S0 Gaussians (mngmm.train).
    In: z_pool, ids, params0, fit, stop; z_test1, true_test1_y, lut (log lines only). Out: params_f."""
    clf, a, off = _clf(st), st["args"], st["offset"]
    _component(st["stop"])(clf, a)
    clf._set_label_offset(off)
    clf.global_params, clf.n_seen = _jnp(st["params0"]), off + a.increment
    p = _component(st["fit"])(clf, st["z_pool"], (st["ids"] + off).astype(int), st["z_test1"],
                              _rel(st, "true_test1_y"), "stage_1_Flearning", False)
    return dict(params_f=_np(p))

det old slots (params0): [0.2425 0.4325 0.3215 0.4204 0.4365] | mean 0.3707
det new slots (params_f, step 1000): [0.3046 0.3789 0.1523 0.1761 0.1402] | mean 0.2304
pass-F threshold 0.1853 -> stop: False | pass-S threshold 0.5242 -> stop: True


- **Kết quả**:
  - `s11_gauss_f`: nhãn là `ids + offset` (5–9) cho cả 748 ảnh pool, kể cả 50 ảnh cũ; khởi tạo từ `params0`; gọi `fit_svi` với `use_correct_scaling_factor=False`, tức nhánh F của luật dừng. Nhãn test chỉ để in log.
  - det của khe cũ (`params0`): 0.2425–0.4365, trung bình 0.3707. det của khe mới sau 1000 bước: cụm 0 (inclusion) 0.3046, cụm 1 (waist) 0.3789, ba cụm silk 0.1523, 0.1761, 0.1402; trung bình 0.2304.
  - Ngưỡng F 0.1853: không dừng. Ngưỡng S 0.5242: đã vượt.
- **Ý nghĩa**:
  - Dự đoán đúng ở ngưỡng và ở hai kết luận dừng/không dừng. Sai ở mức det: trung bình 0.230, thấp hơn khoảng dự đoán 0.3–0.4 và thấp hơn cả khe cũ.
  - Ba khe của cụm silk có det thấp nhất (0.14–0.18).
  - [SUY LUẬN] Với cùng ngân sách bước, det của khe mới đi theo hình dạng dữ liệu của cụm: cụm silk chặt hơn trong không gian PCA-64, nên Σ co nhiều hơn. Mục 3 kiểm điều này bằng phương sai trong cụm.
  - Lượt F chạy hết 1000 bước chỉ vì nhánh F đặt ngưỡng ở một nửa det cũ. Với luật của lượt S, lượt F đã dừng trước bước 1000.

## 3. `params_f` so với dữ liệu

### 3.1 Khe mới so với cụm của nó, trên 15 ảnh chụp

Với mỗi khe mới k (cụm k của bước 09), trong không gian PCA-64 (`z_pool`):
- n_k: số ảnh pool của cụm;
- v_k: phương sai trung bình theo 64 chiều của các ảnh trong cụm, và số chiều có phương sai lớn hơn 1;
- cos(μ_k, x̄_k): cosine giữa μ học được và tâm của cụm;
- det Σ_k ở `params_f`.

In bảng của r0 s4. Với 15 ảnh chụp, in det trung bình của khe mới, det nhỏ nhất, và tương quan hạng Spearman giữa v_k và det Σ_k trên 75 khe.

Dự đoán [SUY LUẬN]:
- cos(μ_k, x̄_k) khoảng 0.99, như r_μ 0.992–0.996 của khe cũ ở `06` §3.1;
- Spearman(v_k, det Σ_k) dương và lớn (≥ 0.7): cụm càng tản thì Σ càng ít co;
- det trung bình của khe mới thấp hơn det trung bình của khe cũ ở cả 15 ảnh chụp.

In [4]:
# 3.1 new slots of params_f vs the data of their cluster (PCA-64 space), 15 snapshots
import pandas as pd
from scipy.stats import spearmanr

RUNS = [f"gc10_v2_ft_r{r}_s{s}" for r in range(3) for s in range(5)]
rows, slots = [], []
for run in RUNS:
    b = replay.snapshots(run)
    z, ids = np.asarray(b["05"]["out"]["z_pool"]), np.asarray(b["09"]["out"]["ids"])
    P0, PF = b["06"]["out"]["params0"], b["11"]["out"]["params_f"]
    do, dn = det(P0["class_covs"][:5]), det(PF["class_covs"][5:])
    for k in range(5):
        x = z[ids == k]; mu = PF["class_means"][5 + k]; xb = x.mean(axis=0); v = x.var(axis=0)
        slots.append(dict(run=run.replace("gc10_v2_ft_", ""), k=k, n=len(x), v_mean=v.mean(), dims_v_gt1=int((v > 1).sum()),
                          cos_mu_center=float(mu @ xb / np.linalg.norm(mu) / np.linalg.norm(xb)),
                          max_abs_mu=float(np.abs(mu).max()), det=float(dn[k])))
    rows.append(dict(run=run.replace("gc10_v2_ft_", ""), det_old_mean=do.mean(), det_new_mean=dn.mean(), det_new_min=dn.min()))
S31, T31 = pd.DataFrame(slots), pd.DataFrame(rows).set_index("run")
print("r0 s4, new slots:")
print(S31[S31.run == "r0_s4"].round(4).to_string(index=False))
print("\n15 snapshots:")
print(T31.round(4).to_string())
r, p = spearmanr(S31.v_mean, S31.det)
print(f"\nSpearman(v_mean, det) over {len(S31)} slots: {r:+.2f} (p {p:.1e}) | cos range {S31.cos_mu_center.min():.3f}-{S31.cos_mu_center.max():.3f} | "
      f"new mean < old mean in {int((T31.det_new_mean < T31.det_old_mean).sum())}/15")

r0 s4, new slots:
  run  k   n  v_mean  dims_v_gt1  cos_mu_center  max_abs_mu    det
r0_s4  0  95  1.5312          25         0.6184      0.0319 0.3046
r0_s4  1 157  1.8431          30         0.8049      0.0315 0.3789
r0_s4  2 272  1.0024          21         0.7335      0.0323 0.1523
r0_s4  3 121  0.4499           6         0.7502      0.0324 0.1761
r0_s4  4 103  0.2188           0         0.7546      0.0323 0.1402

15 snapshots:
       det_old_mean  det_new_mean  det_new_min
run                                           
r0_s0        0.3708        0.2779       0.1509
r0_s1        0.3704        0.2820       0.1491
r0_s2        0.3723        0.2631       0.1579
r0_s3        0.3719        0.2245       0.1466
r0_s4        0.3707        0.2304       0.1402
r1_s0        0.4192        0.2838       0.1477
r1_s1        0.4174        0.2901       0.1479
r1_s2        0.4206        0.2888       0.1621
r1_s3        0.4188        0.2432       0.1399
r1_s4        0.4194        0.2332       0.1476
r

- **Kết quả**:
  - r0 s4: max|μ| của 5 khe mới 0.0315–0.0324. cos(μ_k, x̄_k) 0.618–0.805. Hai cụm silk thuần (3, 4) có phương sai trung bình 0.45 và 0.22, chỉ 6 và 0 chiều có phương sai > 1; det của chúng 0.176 và 0.140. Cụm waist (1) có phương sai 1.84, 30 chiều > 1, det 0.379.
  - 15 ảnh chụp: det trung bình của khe mới 0.22–0.29, thấp hơn det trung bình của khe cũ (0.37–0.43) ở 15/15. det nhỏ nhất 0.14–0.17.
  - Spearman(phương sai trong cụm, det) trên 75 khe: +0.89 (p 7e-27). cos(μ, tâm cụm) 0.618–0.875.
- **Ý nghĩa**:
  - Đúng như dự đoán ở tương quan (+0.89 ≥ 0.7) và ở det mới < det cũ (15/15). Sai ở cos: dự đoán 0.99 là vì mình đọc nhầm r_μ của `06` §3.1 thành một cosine. Chính `06` đo cos giữa μ và tâm lớp của khe cũ là 0.63–0.75; khe mới ở đây nằm trong cùng khoảng (0.62–0.88).
  - [SUY LUẬN] μ của khe mới gần như chạm ngân sách ở mọi toạ độ (0.0315–0.0324), nên hướng của nó gần với dấu của tâm cụm hơn là với chính tâm cụm. Đây là D23 lặp lại ở lượt F.
  - [SUY LUẬN] det của một khe đi theo độ tản của cụm. Lượt S dừng khi det trung bình của khe mới xuống dưới 0.524, nên các cụm silk chặt làm det giảm nhanh nhất. Vì vậy số cụm silk ở bước 09 và độ chặt của chúng có thể quyết định bước dừng của lượt S (`07` §3.1 thấy bước dừng đi theo nghiệm của 09). Kiểm ở 13.

## 4. Cột 11 của sổ mất điểm (phản thực tế)

### 4.1 Nếu pipeline dừng sau lượt F

Gọi `score.ledger` (cột 09, 10, 11) cho 15 ảnh chụp. Cột 11: bộ phân lớp với `params_f`, cả 10 khe, đặt ảnh test lớp mới vào đúng khe của cụm mang tên lớp đó. In `ledger_summary` của r0 s4, rồi balanced của cột 10 và 11 theo ảnh chụp và theo FT seed.

Từ 10 sang 11, kết quả đổi theo hai đường: ảnh bị khe cũ "cướp" (khe cũ có q mạnh, `08` §3), và ảnh được bộ phân lớp Gaussian đặt khác với k-means (khoảng cách Euclid tới tâm ở 768 chiều thô thay bằng log-mật độ ở PCA-64).

Dự đoán [SUY LUẬN]:
- balanced cột 11 thấp hơn cột 10 ở phần lớn ảnh chụp; ở r0 s4 khoảng 0.33–0.40 (cột 10: 0.395);
- mất nhiều nhất ở inclusion (Q08.02: ảnh inclusion bị đẩy sang lớp cũ) và một phần ở silk; được thêm ít (dưới 10 ảnh mỗi ảnh chụp), vì khe rolled và crease học từ hai cụm silk.

In [5]:
# 4.1 loss ledger with column 11 (pass-F classifier, counterfactual), 15 snapshots
import json, time
import score
split = json.load(open(ROOT / "splits" / "gc10_split.json"))
names = [c.split("_")[0] for c in split["classes"]]
t0 = time.time()
LEDGER = {run: score.ledger(run) for run in RUNS}
print(f"ledger of 15 snapshots: {time.time() - t0:.0f} s")
print("\nr0 s4, test1:")
print(score.ledger_summary(LEDGER[RUN]["test1"], names, ["09", "10", "11"]).round(3).to_string())
bal = pd.DataFrame({run.replace("gc10_v2_ft_", ""): score.ledger_summary(LEDGER[run]["test1"], names, ["10", "11"]).loc["balanced", ["10", "11"]]
                    for run in RUNS}).T
bal["ft"] = [i[:2] for i in bal.index]
print("\nbalanced novel, column 10 vs 11:")
print(bal.round(3).to_string())
print("\nmean by FT seed:")
print(bal.groupby("ft").mean().round(3).to_string(), "\n11 < 10 in", int((bal["11"] < bal["10"]).sum()), "/ 15")

ledger of 15 snapshots: 2 s

r0 s4, test1:
               n       09      10  lost_09_10  gained_09_10     11  lost_10_11  gained_10_11
silk       124.0  116.000  86.000        30.0           0.0  88.00         0.0           2.0
inclusion   43.0   15.000  15.000         0.0           0.0  10.00         5.0           0.0
rolled       6.0    0.000   0.000         0.0           0.0   0.00         0.0           0.0
crease      11.0    0.000   0.000         0.0           0.0   1.00         0.0           1.0
waist       30.0   28.000  28.000         0.0           0.0  26.00         2.0           0.0
balanced     NaN    0.444   0.395         NaN           NaN   0.38         NaN           NaN

balanced novel, column 10 vs 11:
          10     11  ft
r0_s0  0.503  0.459  r0
r0_s1  0.498  0.462  r0
r0_s2  0.503  0.431  r0
r0_s3  0.390  0.373  r0
r0_s4  0.395  0.380  r0
r1_s0  0.398  0.295  r1
r1_s1  0.394  0.320  r1
r1_s2  0.398  0.298  r1
r1_s3  0.369  0.333  r1
r1_s4  0.368  0.309  r1
r2_s0  0

- **Kết quả** (code `ae132c1`):
  - Sổ mất điểm của 15 ảnh chụp tính trong 2 s.
  - r0 s4, từ cột 10 sang 11:
    - silk 86 → 88 (được 2), inclusion 15 → 10 (mất 5), waist 28 → 26 (mất 2), crease 0 → 1 (được 1), rolled 0 → 0;
    - balanced 0.395 → 0.380.
  - Balanced cột 11 thấp hơn cột 10 ở 15/15 ảnh chụp (r2 s4 bằng nhau sau khi làm tròn tới 3 chữ số). Trung bình theo FT seed, cột 10 → 11: r0 0.458 → 0.421, r1 0.386 → 0.311, r2 0.405 → 0.386.
- **Ý nghĩa**:
  - Đúng như dự đoán: cột 11 thấp hơn ở mọi ảnh chụp; r0 s4 0.380 nằm trong khoảng 0.33–0.40; inclusion mất nhiều nhất; được thêm ít (3 ảnh ở r0 s4).
  - Chuyển từ phép gán k-means sang bộ phân lớp Gaussian của lượt F làm mất 3.7 / 7.5 / 1.9 điểm (r0 / r1 / r2). r1 mất nhiều nhất, lần thứ hai sau bước đặt tên (`10-naming` §3).
  - [SUY LUẬN] Cột 11 của r0, r1, r2 (0.421 / 0.311 / 0.386) đã gần mức cuối pipeline mà E03.02 đo (0.394 / 0.292 / 0.362, cộng 5 seed). Nếu 14 xác nhận, phần lớn chỗ mất sau bước 10 đã xảy ra ở lượt F, và bộ lọc cùng lượt S chỉ đổi thêm khoảng 2–3 điểm.

### 4.2 Ảnh mất từ cột 10 sang 11 rơi vào khe nào

Với mỗi ảnh test lớp mới đúng ở cột 10 nhưng sai ở cột 11, tìm khe mà bộ phân lớp của lượt F chọn: một khe cũ (0–4, tức ảnh bị lớp cũ "cướp") hay một khe mới khác. Đếm theo lớp thật và theo FT seed; với khe cũ, ghi lớp cũ nào. Khe dự đoán lấy bằng `_predict` của pipeline, như trong `score.ledger`.

Dự đoán [SUY LUẬN]:
- phần lớn (≥ 70%) ảnh mất rơi vào khe cũ. Khe cũ đã qua 1000 bước ở S0 với cùng ngân sách như khe mới, nhưng học từ dữ liệu có nhãn của lớp cũ;
- inclusion chiếm phần lớn số ảnh mất, sang water_spot hoặc oil_spot (Q08.02, `00-data-gc10` §3.4);
- ở r1, số ảnh mất cao hơn hẳn r0 và r2, và cũng chủ yếu sang khe cũ.

In [6]:
# 4.2 where the images lost between column 10 and 11 go: an old slot (0-4) or another novel slot
import jax.numpy as jnp
from collections import Counter

rows42, to_old = [], Counter()
for run in RUNS:
    st = replay.load(run, "11", check=False)
    replay.predict_default(st["clf"])
    slot = np.asarray(st["clf"]._predict(jnp.asarray(st["z_test1"]), replay._jnp(st["params_f"]))[0])
    t = LEDGER[run]["test1"]
    lost = ((t["10"] == 1) & (t["11"] == 0)).to_numpy()
    for y, s in zip(t["y"].to_numpy()[lost], slot[lost]):
        rows42.append(dict(run=run.replace("gc10_v2_ft_", ""), ft=run[11:13], cls=names[y], to_old=bool(s < 5)))
        if s < 5:
            to_old[(names[y], names[s])] += 1
T42 = pd.DataFrame(rows42)
print("lost 10 -> 11 per FT seed (images summed over 5 snapshots), share sent to an old slot:")
print(T42.groupby("ft").agg(lost=("to_old", "size"), to_old=("to_old", "sum"), share_old=("to_old", "mean")).round(2).to_string())
print("\nlost per class and FT seed:")
print(T42.pivot_table(index="cls", columns="ft", values="to_old", aggfunc="size", fill_value=0).to_string())
print("\nold slot taking them (true -> old slot), 15 snapshots:", dict(to_old.most_common(8)))

lost 10 -> 11 per FT seed (images summed over 5 snapshots), share sent to an old slot:
    lost  to_old  share_old
ft                         
r0    42       8       0.19
r1   114      28       0.25
r2    51      15       0.29

lost per class and FT seed:
ft         r0  r1  r2
cls                  
crease     10   4   0
inclusion  19  54  26
silk        7  56  25
waist       6   0   0

old slot taking them (true -> old slot), 15 snapshots: {('inclusion', 'oil'): 35, ('inclusion', 'punching'): 9, ('crease', 'welding'): 3, ('crease', 'oil'): 3, ('crease', 'crescent'): 1}


- **Kết quả** (cộng 5 ảnh chụp mỗi FT seed):
  - Số ảnh mất từ cột 10 sang 11: r0 42, r1 114, r2 51. Phần sang khe cũ chỉ 0.19 / 0.25 / 0.29; phần còn lại (71–81%) sang một khe mới khác.
  - Theo lớp: inclusion mất 19 / 54 / 26, silk 7 / 56 / 25, crease 10 / 4 / 0, waist 6 / 0 / 0.
  - Ảnh bị khe cũ cướp, trên 15 ảnh chụp: inclusion → oil_spot 35, inclusion → punching_hole 9, crease → welding_line 3, crease → oil_spot 3, crease → crescent_gap 1.
- **Ý nghĩa**:
  - Dự đoán sai ở ý chính: phần lớn ảnh mất không bị lớp cũ cướp mà được bộ phân lớp Gaussian chuyển sang một khe mới khác. Sai cả ở lớp cũ nhận ảnh: inclusion bị oil_spot cướp, không phải water_spot. Đúng ở inclusion chiếm nhiều ảnh mất, và ở r1 mất nhiều nhất.
  - [SUY LUẬN] Lý do chính của bước 10 → 11 là bộ phân lớp Gaussian (log-mật độ ở PCA-64, chủ yếu do số hạng bậc hai, `08` §3) chia lại ảnh test giữa các khe mới khác với k-means (khoảng cách Euclid ở 768 chiều thô). Ở r1 phần chia lại này lớn nhất (110 ảnh inclusion và silk).

## Summary (EN)

1. **Step 11 replays bit for bit, and pass F leaves the old slots untouched.** `rerun` gives `params_f` identical to the snapshot (1000 SVI steps, 59 s). Pass F trains the 5 novel slots on all 748 pool images with cluster ids as labels, starting from `params0`. The 5 old slots stay bit-identical to `params0`, since no pool image carries an old label. Pass F never stops: its threshold is half the old mean det (0.185 at r0 s4), and the new slots end at a mean det of 0.230, below the pass-S threshold of 0.524 (§1.1, §2.1).
2. **The new slots use the full step budget, and their det follows the spread of their cluster.** max|μ| of the new slots is 0.0315–0.0324 (budget 0.0334). Over 75 slots, the Spearman correlation between det Σ and the within-cluster variance is +0.89. Tight silk clusters reach det 0.14–0.18. The new mean det (0.22–0.29) is below the old one (0.37–0.43) in 15/15 snapshots. cos(μ, cluster centre) is 0.62–0.88, as for the old slots at S0 (§3.1). [inference] Through this det, the number and tightness of silk clusters can set the pass-S stop step; checked at step 13.
3. **Stopping after pass F already loses points (ledger column 11, counterfactual).** Cluster-level balanced novel falls from column 10 to 11 in 15/15 snapshots: 0.458 → 0.421 (r0), 0.386 → 0.311 (r1), 0.405 → 0.386 (r2) (§4.1).
4. **The loss is a reshuffle among the novel slots, not old-class stealing.** Of the images lost between columns 10 and 11, 71–81% go to another novel slot. Old slots take 19–29%, mainly inclusion → oil_spot (35 images over 15 snapshots). r1 loses the most (114 images, mostly inclusion and silk) (§4.2). [inference] The Gaussian classifier (PCA-64, quadratic term) splits the test set differently from k-means (raw 768-d); this is a second r1-specific loss after the naming of step 10.